# N-BEATS


In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


Mounted at /content/drive
/content/drive/MyDrive/ml_final_project


In [2]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project

drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")

import feature_pipeline
import importlib
importlib.reload(feature_pipeline)
from feature_pipeline import load_raw_data, run_feature_pipeline

from metrics import wmae


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q mlflow dagshub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [4]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)
mlflow.set_experiment('NBEATS_Training')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f030d86a-2b3a-448c-9020-eb2f26e05136&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f68001bc2c13b547725534d1103984d3c1e222df43900d90adec635a66779259




Accessing as Saba0033

Initialized MLflow to track repo "aochi23/ml_final_project"

Repository aochi23/ml_final_project initialized!

2026/07/11 10:37:20 INFO mlflow.tracking.fluent: Experiment with name 'NBEATS_Training' does not exist. Creating a new experiment.


cuda


## Data


In [5]:
train, test, features, stores = load_raw_data(path=f'{drive_repo}/data/')
full_df = run_feature_pipeline(train, test, features, stores)

train_df = full_df[full_df['is_train'] == 1].drop(columns=['is_train'])
print(train_df.shape)


(421570, 39)


In [6]:
# group by store-dept
pairs = train_df.groupby(['Store', 'Dept'])
print(f'total pairs: {len(pairs)}')

LOOKBACK = 12
HORIZON = 1
MIN_LEN = LOOKBACK + HORIZON + 4

series_list = []
pair_keys = []

for (store, dept), grp in pairs:
    grp = grp.sort_values('Date')
    sales = grp['Weekly_Sales'].values
    holidays = grp['IsHoliday'].values
    if len(sales) < MIN_LEN:
        continue
    series_list.append((sales, holidays))
    pair_keys.append((store, dept))

print(f'pairs with enough data: {len(series_list)}')


total pairs: 3331
pairs with enough data: 3108


In [7]:
with mlflow.start_run(run_name='NBEATS_Preprocessing'):
    mlflow.log_param('lookback', LOOKBACK)
    mlflow.log_param('horizon', HORIZON)
    mlflow.log_param('pairs_kept', len(series_list))
    print('preprocessing logged')


preprocessing logged
🏃 View run NBEATS_Preprocessing at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/98940cb84d314d2fb70ea261580d72ca
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7


## Dataset


In [8]:
class SalesDataset(Dataset):
    def __init__(self, series_list, lookback, horizon, train=True, val_weeks=12):
        self.samples = []
        self.holidays = []
        for sales, hols in series_list:
            n = len(sales)
            if train:
                end = n - val_weeks
            else:
                end = n

            start_from = 0 if train else n - val_weeks - lookback
            start_from = max(start_from, 0)

            for i in range(start_from, end - lookback - horizon + 1):
                x = sales[i:i + lookback]
                y = sales[i + lookback:i + lookback + horizon]
                h = hols[i + lookback:i + lookback + horizon]
                if not train and i + lookback < n - val_weeks:
                    continue
                self.samples.append((x, y))
                self.holidays.append(h)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)


In [9]:
VAL_WEEKS = 12

train_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=True, val_weeks=VAL_WEEKS)
val_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=False, val_weeks=VAL_WEEKS)

print(f'train samples: {len(train_ds)}')
print(f'val samples: {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2048, shuffle=False)


train samples: 345774
val samples: 37118


## Model


In [10]:
class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden, share_thetas=False):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.backcast_fc = nn.Linear(hidden, input_size)
        self.forecast_fc = nn.Linear(hidden, theta_size)

    def forward(self, x):
        h = self.fc(x)
        return self.backcast_fc(h), self.forecast_fc(h)


class NBeats(nn.Module):
    def __init__(self, lookback, horizon, n_stacks=2, n_blocks=3, hidden=256):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.blocks = nn.ModuleList()
        for _ in range(n_stacks):
            for _ in range(n_blocks):
                self.blocks.append(NBeatsBlock(lookback, horizon, hidden))

    def forward(self, x):
        forecast = torch.zeros(x.shape[0], self.horizon, device=x.device)
        backcast = x
        for block in self.blocks:
            b, f = block(backcast)
            backcast = backcast - b
            forecast = forecast + f
        return forecast


## Training


In [11]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    return total_loss / n


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    n = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            total_loss += loss.item() * x.size(0)
            n += x.size(0)
    return total_loss / n


In [12]:
def compute_val_wmae(model, val_ds):
    model.eval()
    all_preds = []
    all_true = []
    all_hols = []
    with torch.no_grad():
        for i in range(len(val_ds)):
            x, y = val_ds[i]
            pred = model(x.unsqueeze(0).to(device))
            all_preds.append(pred.cpu().numpy().flatten())
            all_true.append(y.numpy().flatten())
            all_hols.append(val_ds.holidays[i])

    preds = np.concatenate(all_preds)
    true = np.concatenate(all_true)
    hols = np.concatenate(all_hols)
    return wmae(true, preds, hols)


In [13]:
PARAMS = {
    'n_stacks': 2,
    'n_blocks': 3,
    'hidden': 256,
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'lr': 1e-3,
    'epochs': 30,
    'batch_size': 1024,
}

model = NBeats(
    lookback=PARAMS['lookback'],
    horizon=PARAMS['horizon'],
    n_stacks=PARAMS['n_stacks'],
    n_blocks=PARAMS['n_blocks'],
    hidden=PARAMS['hidden'],
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.L1Loss()

print(f'params: {sum(p.numel() for p in model.parameters()):,}')


params: 1,224,270


In [14]:
with mlflow.start_run(run_name='NBEATS_Baseline'):
    mlflow.log_params(PARAMS)

    best_val_loss = float('inf')
    best_state = None

    for epoch in range(PARAMS['epochs']):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss = eval_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        mlflow.log_metric('train_mae', train_loss, step=epoch)
        mlflow.log_metric('val_mae', val_loss, step=epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0:
            print(f'epoch {epoch+1}: train_mae={train_loss:.2f}, val_mae={val_loss:.2f}')

    model.load_state_dict(best_state)
    model.to(device)
    val_wmae_score = compute_val_wmae(model, val_ds)

    mlflow.log_metric('best_val_mae', best_val_loss)
    mlflow.log_metric('val_wmae', val_wmae_score)
    print(f'\nbest val MAE: {best_val_loss:.2f}')
    print(f'val WMAE: {val_wmae_score:.2f}')


epoch 5: train_mae=1667.53, val_mae=1235.70
epoch 10: train_mae=1607.52, val_mae=1198.08
epoch 15: train_mae=1539.20, val_mae=1221.90
epoch 20: train_mae=1504.02, val_mae=1218.93
epoch 25: train_mae=1444.97, val_mae=1217.19
epoch 30: train_mae=1393.66, val_mae=1207.56

best val MAE: 1186.64
val WMAE: 1267.65
🏃 View run NBEATS_Baseline at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/a6250feb361d4cdf85013f27d5573d6a
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7


## HPO


In [15]:
configs = [
    {'n_stacks': 2, 'n_blocks': 3, 'hidden': 256, 'lr': 1e-3},
    {'n_stacks': 3, 'n_blocks': 3, 'hidden': 256, 'lr': 1e-3},
    {'n_stacks': 2, 'n_blocks': 3, 'hidden': 512, 'lr': 5e-4},
    {'n_stacks': 2, 'n_blocks': 4, 'hidden': 256, 'lr': 1e-3},
]

hpo_results = []

with mlflow.start_run(run_name='NBEATS_HPO'):
    for i, cfg in enumerate(configs):
        with mlflow.start_run(run_name=f'NBEATS_HPO_trial{i}', nested=True):
            mlflow.log_params(cfg)

            m = NBeats(
                lookback=LOOKBACK, horizon=HORIZON,
                n_stacks=cfg['n_stacks'],
                n_blocks=cfg['n_blocks'],
                hidden=cfg['hidden'],
            ).to(device)

            opt = torch.optim.Adam(m.parameters(), lr=cfg['lr'])
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
            crit = nn.L1Loss()

            best_vl = float('inf')
            best_st = None
            for epoch in range(25):
                tl = train_epoch(m, train_loader, opt, crit)
                vl = eval_epoch(m, val_loader, crit)
                sched.step(vl)
                if vl < best_vl:
                    best_vl = vl
                    best_st = {k: v.cpu().clone() for k, v in m.state_dict().items()}

            m.load_state_dict(best_st)
            m.to(device)
            score = compute_val_wmae(m, val_ds)

            mlflow.log_metric('best_val_mae', best_vl)
            mlflow.log_metric('val_wmae', score)
            hpo_results.append({**cfg, 'val_wmae': score, 'val_mae': best_vl})
            print(f'trial {i}: wmae={score:.2f}, mae={best_vl:.2f}, cfg={cfg}')

hpo_df = pd.DataFrame(hpo_results).sort_values('val_wmae')
hpo_df


trial 0: wmae=1268.20, mae=1186.71, cfg={'n_stacks': 2, 'n_blocks': 3, 'hidden': 256, 'lr': 0.001}
🏃 View run NBEATS_HPO_trial0 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/2f88aa58b4204e4b8daacdfdc471eb3f
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7
trial 1: wmae=1272.88, mae=1187.78, cfg={'n_stacks': 3, 'n_blocks': 3, 'hidden': 256, 'lr': 0.001}
🏃 View run NBEATS_HPO_trial1 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/e6fec0f1a6ba4c3e9c6c06bee1752adc
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7
trial 2: wmae=1263.22, mae=1187.44, cfg={'n_stacks': 2, 'n_blocks': 3, 'hidden': 512, 'lr': 0.0005}
🏃 View run NBEATS_HPO_trial2 at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/ba490bc61cda49ac83aab2ff6ae09c76
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7
trial 3: wmae=1

,n_stacks,n_blocks,hidden,lr,val_wmae,val_mae
2,2,3,512,0.0005,1263.220637,1187.443280
3,2,4,256,0.0010,1265.154659,1191.090366
0,2,3,256,0.0010,1268.195657,1186.708341
1,3,3,256,0.0010,1272.879558,1187.776188


## Final model


In [16]:
best_cfg = hpo_df.iloc[0]

FINAL_PARAMS = {
    'n_stacks': int(best_cfg['n_stacks']),
    'n_blocks': int(best_cfg['n_blocks']),
    'hidden': int(best_cfg['hidden']),
    'lookback': LOOKBACK,
    'horizon': HORIZON,
    'lr': float(best_cfg['lr']),
    'epochs': 40,
}

# retrain on all data
all_ds = SalesDataset(series_list, LOOKBACK, HORIZON, train=True, val_weeks=0)
all_loader = DataLoader(all_ds, batch_size=1024, shuffle=True)

final_model = NBeats(
    lookback=FINAL_PARAMS['lookback'],
    horizon=FINAL_PARAMS['horizon'],
    n_stacks=FINAL_PARAMS['n_stacks'],
    n_blocks=FINAL_PARAMS['n_blocks'],
    hidden=FINAL_PARAMS['hidden'],
).to(device)

opt = torch.optim.Adam(final_model.parameters(), lr=FINAL_PARAMS['lr'])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
crit = nn.L1Loss()

with mlflow.start_run(run_name='NBEATS_Final') as run:
    mlflow.log_params(FINAL_PARAMS)

    for epoch in range(FINAL_PARAMS['epochs']):
        tl = train_epoch(final_model, all_loader, opt, crit)
        sched.step(tl)
        mlflow.log_metric('train_mae', tl, step=epoch)
        if (epoch + 1) % 10 == 0:
            print(f'epoch {epoch+1}: train_mae={tl:.2f}')

    mlflow.log_metric('val_wmae', float(best_cfg['val_wmae']))

    torch.save(final_model.state_dict(), f'{drive_repo}/nbeats_final.pt')
    mlflow.log_artifact(f'{drive_repo}/nbeats_final.pt')

    import sys
    if f'{drive_repo}/src' not in sys.path:
        sys.path.append(f'{drive_repo}/src')
    from walmart_pytorch import WalmartPyTorchPipeline
    import mlflow.sklearn

    pipeline = WalmartPyTorchPipeline(model=final_model, lookback=LOOKBACK, device=device)
    pipeline.set_raw_data(train, features, stores)

    mlflow.sklearn.log_model(
        pipeline,
        name='pytorch_pipeline',
        registered_model_name='Walmart_NBEATS',
        code_paths=[f'{drive_repo}/src/walmart_pytorch.py'],
            serialization_format='cloudpickle'
    )
    print(f'model saved, best hpo wmae = {best_cfg["val_wmae"]:.2f}')


epoch 10: train_mae=1549.46
epoch 20: train_mae=1472.20
epoch 30: train_mae=1394.42
epoch 40: train_mae=1292.47


2026/07/11 11:25:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/11 11:25:34 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'Walmart_NBEATS'.
2026/07/11 11:25:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_NBEATS, version 1
Cr

model saved, best hpo wmae = 1263.22
🏃 View run NBEATS_Final at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7/runs/ec5b2607f2284d54b02a2a7042f68fca
🧪 View experiment at: https://dagshub.com/aochi23/ml_final_project.mlflow/#/experiments/7


## Register model


In [17]:
model_uri = 'models:/Walmart_NBEATS/latest'
loaded = mlflow.sklearn.load_model(model_uri)
raw_test = test[['Store', 'Dept', 'Date', 'IsHoliday']].head(100)
preds = loaded.predict(raw_test)
print(f'Raw test predict ok, got {len(preds)} predictions')


Raw test predict ok, got 100 predictions
